# Faster R-CNN: detección de peatones con Region Proposal Networks

**Momento Evaluativo — Aplicaciones del aprendizaje profundo**
Institución Universitaria ITM · Introducción al aprendizaje profundo · Docente: Kevin Osorno Castillo

**Aplicación:** Detección de objetos (Anexo A, punto 5)
**Artículo:** Ren, S., He, K., Girshick, R., & Sun, J. (2015). *Faster R-CNN: Towards Real-Time Object Detection with Region Proposal Networks*. NeurIPS 2015. [arXiv:1506.01497](https://arxiv.org/abs/1506.01497)
**Dataset:** [Penn-Fudan Pedestrian](https://www.cis.upenn.edu/~jshi/ped_html/) (170 imágenes de peatones con máscaras de instancia)

---

### Objetivo del proyecto
Adaptar un detector Faster R-CNN preentrenado (COCO) a la tarea de **detección de peatones** y analizar sus predicciones de clases y *bounding boxes*, incluyendo **errores de localización y falsos positivos**, con las métricas **Precision, Recall, IoU y mAP** (experimento sugerido en el Anexo A).

### Experimentos
| ID | Configuración | Pregunta que responde |
|---|---|---|
| **E0** | Faster R-CNN R50-FPN COCO, *zero-shot* (solo clase `person`) | ¿Qué tan bien detecta peatones un modelo que nunca vio este dataset? |
| **E1** | R50-FPN con **extractor congelado**: solo se entrenan RPN + cabeza Fast R-CNN | ¿Basta con reentrenar las capas exclusivas (*transfer learning* como extractor)? |
| **E2** | R50-FPN con **fine-tuning** (últimas 3 etapas del backbone + RPN + cabeza), entrenamiento conjunto aproximado | ¿Cuánto aporta ajustar las capas compartidas? |
| **E3** | MobileNetV3-Large-FPN con fine-tuning | *Trade-off* velocidad vs. precisión del extractor (análogo a ZF vs. VGG del paper) |
| **E4** | R50-FPN con el **entrenamiento alterno de 4 pasos** del paper (Sec. 3.2) | ¿El esquema original del artículo produce resultados comparables al entrenamiento conjunto? |

Además se realizan análisis de caja blanca (anclas, salidas de la RPN, codificación de cajas), análisis de errores, ablación del número de propuestas y medición del tiempo de inferencia por componente.

### Contenido
1. Configuración del entorno
2. Datos: descarga, *dataset* de PyTorch, partición y análisis exploratorio
3. Faster R-CNN por dentro: backbone, anclas, RPN y cabeza Fast R-CNN
4. Métricas: IoU, Precision, Recall, AP / mAP y taxonomía de errores
5. Entrenamiento y experimentos (E0–E4)
6. Resultados cuantitativos
7. Resultados cualitativos y análisis de errores
8. Complejidad computacional y número de propuestas
9. Conclusiones
10. Referencias

> **Ejecución recomendada:** Google Colab con GPU (`Entorno de ejecución → Cambiar tipo de entorno → T4 GPU`). Tiempo total aproximado: 25–35 min. Todas las figuras y tablas se guardan en la carpeta `results/` para usarlas en la presentación.

## 1. Configuración del entorno

Se usa **PyTorch + torchvision**, el mismo *stack* utilizado en el curso. Colab ya trae todas las dependencias instaladas. Para una prueba rápida del flujo completo (pocas imágenes, 1 época, imágenes pequeñas) se puede poner `QUICK_RUN = True`.

In [ ]:
import copy
import json
import math
import os
import random
import time
import urllib.request
import zipfile
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import tv_tensors
from torchvision.io import ImageReadMode, read_image
from torchvision.models.detection import (
    FasterRCNN_MobileNet_V3_Large_FPN_Weights,
    FasterRCNN_ResNet50_FPN_Weights,
    fasterrcnn_mobilenet_v3_large_fpn,
    fasterrcnn_resnet50_fpn,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.transform import resize_boxes
from torchvision.ops import box_iou, masks_to_boxes
from torchvision.transforms import v2 as T
from torchvision.transforms.v2 import functional as TF

print("PyTorch:", torch.__version__, "| torchvision:", torchvision.__version__)

In [ ]:
# --- Modo de prueba rápida (pocas imágenes, 1 época). Usar False para los resultados finales ---
QUICK_RUN = os.environ.get("QUICK_RUN", "0") == "1"

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_ROOT = Path("data")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CFG = {
    # Partición (Penn-Fudan no trae partición oficial: se fija con semilla)
    "n_test": 50,
    "n_val": 20,
    # Optimización (valores del tutorial oficial de torchvision / del paper)
    "epochs": 10,
    "epochs_per_step_4pasos": 5,   # E4: épocas en cada uno de los 4 pasos
    "batch_size": 2,
    "lr": 0.005,
    "momentum": 0.9,
    "weight_decay": 5e-4,
    "lr_step_frac": 0.7,            # el LR se divide por 10 al 70 % de las épocas
    "lr_gamma": 0.1,
    # Escala de entrada (el modelo redimensiona el lado corto a min_size)
    "min_size": 800,
    "max_size": 1333,
    # Evaluación
    "iou_thr": 0.5,
    "num_workers": 2 if DEVICE.type == "cuda" else 0,
}
if QUICK_RUN:
    CFG.update(epochs=1, epochs_per_step_4pasos=1, min_size=320, max_size=512,
               subset_train=6, subset_val=4, subset_test=6)


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def sync():
    # Sincroniza la GPU para medir tiempos correctamente
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


def savefig(name):
    plt.savefig(RESULTS_DIR / f"{name}.png", dpi=150, bbox_inches="tight")


seed_everything()
print("Dispositivo:", DEVICE, "| QUICK_RUN:", QUICK_RUN)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Datos: Penn-Fudan Pedestrian

**Penn-Fudan** contiene 170 imágenes urbanas (campus de la Universidad de Pensilvania y de Fudan) con peatones etiquetados mediante máscaras de instancia. La página original del dataset reporta 345 peatones, pero las máscaras de la versión descargable contienen **423 instancias** (ver el análisis exploratorio). A partir de cada máscara se obtiene la *bounding box* del peatón con `masks_to_boxes`. Es un problema de **una sola clase** (peatón) más el fondo, por lo que el detector tendrá `num_classes = 2`.

In [ ]:
URL = "https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip"
DATA_DIR = DATA_ROOT / "PennFudanPed"

if not DATA_DIR.exists():
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    zip_path = DATA_ROOT / "PennFudanPed.zip"
    print("Descargando Penn-Fudan...")
    urllib.request.urlretrieve(URL, zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(DATA_ROOT)
print("Imágenes:", len(list((DATA_DIR / "PNGImages").glob("*.png"))),
      "| Máscaras:", len(list((DATA_DIR / "PedMasks").glob("*.png"))))

In [ ]:
class PennFudanDataset(Dataset):
    # Devuelve (imagen, target) en el formato que espera Faster R-CNN de torchvision:
    #   target = {"boxes": [N, 4] en formato (x1, y1, x2, y2), "labels": [N], ...}

    def __init__(self, root, transforms=None):
        self.root = Path(root)
        self.transforms = transforms
        self.imgs = sorted((self.root / "PNGImages").glob("*.png"))
        self.masks = sorted((self.root / "PedMasks").glob("*.png"))
        assert len(self.imgs) == len(self.masks)

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img = read_image(str(self.imgs[idx]), mode=ImageReadMode.RGB)
        mask = read_image(str(self.masks[idx]))[0]
        obj_ids = torch.unique(mask)[1:]                 # 0 = fondo; cada id es un peatón
        masks = (mask == obj_ids[:, None, None]).to(torch.uint8)
        boxes = masks_to_boxes(masks)

        target = {
            "boxes": tv_tensors.BoundingBoxes(boxes, format="XYXY", canvas_size=img.shape[-2:]),
            "labels": torch.ones(len(obj_ids), dtype=torch.int64),   # 1 = peatón
            "image_id": idx,
        }
        img = tv_tensors.Image(img)
        if self.transforms is not None:
            img, target = self.transforms(img, target)
        return img, target


def get_transform(train):
    t = []
    if train:
        # Aumento de datos (sesión 09): las transformaciones v2 ajustan también las cajas
        t += [T.RandomHorizontalFlip(0.5), T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2)]
    t += [T.ToDtype(torch.float32, scale=True), T.ToPureTensor()]
    return T.Compose(t)


def collate_fn(batch):
    # Las imágenes tienen tamaños distintos: el lote es una tupla de listas
    return tuple(zip(*batch))

In [ ]:
# Partición fija train / val / test (se conserva separada para evitar fuga de información)
full_ds = PennFudanDataset(DATA_DIR)
perm = torch.randperm(len(full_ds), generator=torch.Generator().manual_seed(SEED)).tolist()
test_idx = perm[: CFG["n_test"]]
val_idx = perm[CFG["n_test"]: CFG["n_test"] + CFG["n_val"]]
train_idx = perm[CFG["n_test"] + CFG["n_val"]:]
if QUICK_RUN:
    train_idx, val_idx, test_idx = (train_idx[: CFG["subset_train"]], val_idx[: CFG["subset_val"]],
                                    test_idx[: CFG["subset_test"]])

train_ds = Subset(PennFudanDataset(DATA_DIR, get_transform(train=True)), train_idx)
val_ds = Subset(PennFudanDataset(DATA_DIR, get_transform(train=False)), val_idx)
test_ds = Subset(PennFudanDataset(DATA_DIR, get_transform(train=False)), test_idx)

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,
                          collate_fn=collate_fn, num_workers=CFG["num_workers"])
print(f"Entrenamiento: {len(train_ds)} | Validación: {len(val_ds)} | Prueba: {len(test_ds)}")

### 2.1 Análisis exploratorio

Se analizan el número de peatones por imagen, el **tamaño** y la **relación de aspecto** de las cajas. Estas estadísticas se comparan con la configuración de **anclas** del modelo: el tamaño de las cajas se expresa en la escala a la que el modelo ve la imagen (el lado corto se redimensiona a `min_size` píxeles), y la relación de aspecto como alto/ancho, igual que en el `AnchorGenerator` de torchvision.

In [ ]:
raw_ds = PennFudanDataset(DATA_DIR)
rows = []
for i in range(len(raw_ds)):
    img, tgt = raw_ds[i]
    h, w = img.shape[-2:]
    scale = min(CFG["min_size"] / min(h, w), CFG["max_size"] / max(h, w))   # escala usada por el modelo
    for x1, y1, x2, y2 in tgt["boxes"].tolist():
        bw, bh = x2 - x1, y2 - y1
        rows.append(dict(img=i, img_w=w, img_h=h, w=bw, h=bh, aspect=bh / bw,
                         size_model=math.sqrt(bw * bh) * scale))
boxes_df = pd.DataFrame(rows)
per_img = boxes_df.groupby("img").size()

print(f"Imágenes: {len(raw_ds)} | Peatones: {len(boxes_df)} | Peatones/imagen: "
      f"media {per_img.mean():.2f}, máx {per_img.max()}")
print(f"Tamaño de imagen (px): ancho {boxes_df.img_w.min()}–{boxes_df.img_w.max()}, "
      f"alto {boxes_df.img_h.min()}–{boxes_df.img_h.max()}")
print(f"Relación de aspecto alto/ancho: mediana {boxes_df.aspect.median():.2f} "
      f"(rango {boxes_df.aspect.min():.2f}–{boxes_df.aspect.max():.2f})")
display(boxes_df[["w", "h", "aspect", "size_model"]].describe().round(2))

ANCHOR_SIZES = [32, 64, 128, 256, 512]
ANCHOR_RATIOS = [0.5, 1.0, 2.0]

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
per_img.value_counts().sort_index().plot.bar(ax=axes[0], color="#4C72B0")
axes[0].set(title="Peatones por imagen", xlabel="# peatones", ylabel="# imágenes")
axes[1].hist(boxes_df.size_model, bins=np.geomspace(boxes_df.size_model.min(), 600, 35), color="#55A868")
for s in ANCHOR_SIZES:
    axes[1].axvline(s, color="k", ls="--", lw=1)
axes[1].set(title="Tamaño √(w·h) en la escala del modelo\n(líneas: tamaños de ancla)", xlabel="píxeles", xscale="log")
axes[2].hist(boxes_df.aspect, bins=30, color="#C44E52")
for r in ANCHOR_RATIOS:
    axes[2].axvline(r, color="k", ls="--", lw=1)
axes[2].set(title="Relación de aspecto alto/ancho\n(líneas: relaciones de ancla)", xlabel="alto / ancho")
plt.tight_layout(); savefig("eda_estadisticas"); plt.show()

**Observación:** los peatones son objetos **altos y angostos** (alto/ancho ≈ 2–3), por encima de la relación más alargada de las anclas por defecto (2.0). Las anclas no necesitan coincidir exactamente con los objetos: son *referencias* y la capa de regresión de la RPN aprende a deformarlas ($t_w$, $t_h$). Esto se verifica numéricamente en la sección 3.

In [ ]:
def draw_boxes(ax, boxes, color, ls="-", lw=2, labels=None):
    for k, (x1, y1, x2, y2) in enumerate(boxes.tolist() if torch.is_tensor(boxes) else boxes):
        ax.add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec=color, lw=lw, ls=ls))
        if labels is not None:
            ax.text(x1, y1 - 2, labels[k], color="white", fontsize=8,
                    bbox=dict(facecolor=color, alpha=0.8, pad=1, lw=0))


def show_img(ax, img, title=""):
    ax.imshow(img.permute(1, 2, 0).clamp(0, 1).cpu().numpy())
    ax.set_title(title, fontsize=10)
    ax.axis("off")


eda_ds = PennFudanDataset(DATA_DIR, get_transform(train=False))
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, i in zip(axes.ravel(), np.random.default_rng(SEED).choice(len(eda_ds), 8, replace=False)):
    img, tgt = eda_ds[int(i)]
    show_img(ax, img, f"Imagen {i} — {len(tgt['boxes'])} peatón(es)")
    draw_boxes(ax, tgt["boxes"], "lime")
plt.suptitle("Ejemplos de Penn-Fudan con sus cajas reales (ground truth)", fontsize=14)
plt.tight_layout(); savefig("eda_ejemplos"); plt.show()

## 3. Faster R-CNN por dentro

Faster R-CNN tiene dos módulos que **comparten** el mismo extractor convolucional:

1. **RPN (Region Proposal Network)** — *“dónde mirar”*: una pequeña red totalmente convolucional que desliza una ventana $3\times3$ sobre el mapa de características y, para cada posición, predice para $k$ **anclas** una puntuación de *objectness* (capa `cls`) y 4 desplazamientos (capa `reg`).
2. **Detector Fast R-CNN** — *“qué es”*: recorta las características de cada propuesta con **RoI Pooling** (en torchvision, *RoIAlign* multiescala), las clasifica y refina su caja.

La implementación de torchvision (`fasterrcnn_resnet50_fpn`) añade al diseño original una **FPN** (*Feature Pyramid Network*, Lin et al., 2017): en lugar de 9 anclas (3 escalas × 3 relaciones) en un único mapa, usa 5 mapas (P2–P6) con **un tamaño de ancla por nivel y 3 relaciones de aspecto** ($k=3$ por posición). La idea central del paper —propuestas aprendidas sobre características compartidas— se mantiene intacta.

A continuación se construye el modelo preentrenado en COCO y se inspecciona cada componente.

In [ ]:
def count_params(module, trainable_only=False):
    return sum(p.numel() for p in module.parameters() if (p.requires_grad or not trainable_only))


coco_weights = FasterRCNN_ResNet50_FPN_Weights.COCO_V1
coco_model = fasterrcnn_resnet50_fpn(weights=coco_weights, min_size=CFG["min_size"],
                                     max_size=CFG["max_size"]).to(DEVICE).eval()
COCO_CLASSES = coco_weights.meta["categories"]

comp = pd.DataFrame([
    ("Backbone ResNet-50 (capas convolucionales compartidas)", count_params(coco_model.backbone.body)),
    ("FPN (pirámide de características, compartida)", count_params(coco_model.backbone.fpn)),
    ("RPN (conv 3×3 + capas cls y reg)", count_params(coco_model.rpn)),
    ("Cabeza Fast R-CNN (RoIAlign + 2 FC + cls/reg)", count_params(coco_model.roi_heads)),
], columns=["Componente", "Parámetros"])
comp["%"] = (100 * comp["Parámetros"] / comp["Parámetros"].sum()).round(2)
display(comp)
print("Total:", f"{comp['Parámetros'].sum():,}")
print("\nCabeza de la RPN:\n", coco_model.rpn.head)

**Observación:** la RPN representa una fracción mínima de los parámetros: generar propuestas es casi “gratis” porque reutiliza las características ya calculadas por las capas compartidas. Esa es la contribución central del artículo.

### 3.1 Recorrido de una imagen por la red (paso a paso)
Se ejecuta manualmente cada etapa del `forward` para ver las dimensiones de los tensores: transformación → backbone+FPN → cabeza RPN (`cls`: $k$ canales, `reg`: $4k$ canales) → anclas.

> En el paper la capa `cls` produce $2k$ salidas (softmax de 2 clases); torchvision usa una sigmoide con $k$ salidas, que es matemáticamente equivalente para un problema binario (el propio paper lo menciona como alternativa).

In [ ]:
sample_img, sample_tgt = test_ds[0]
with torch.no_grad():
    images, _ = coco_model.transform([sample_img.to(DEVICE)])         # normaliza y redimensiona
    feats = coco_model.backbone(images.tensors)                       # dict de mapas FPN
    feat_list = list(feats.values())
    objectness, deltas = coco_model.rpn.head(feat_list)               # salidas cls y reg
    anchors = coco_model.rpn.anchor_generator(images, feat_list)      # anclas por imagen

ag = coco_model.rpn.anchor_generator
print("Imagen original:", tuple(sample_img.shape[-2:]), "-> entrada a la red:", tuple(images.image_sizes[0]),
      "(con relleno:", tuple(images.tensors.shape[-2:]), ")\n")
rows = []
for (name, f), o, d, sizes, ratios in zip(feats.items(), objectness, deltas, ag.sizes, ag.aspect_ratios):
    H, W = f.shape[-2:]
    k = o.shape[1]
    rows.append(dict(nivel=name, stride=round(images.tensors.shape[-1] / W), mapa=f"{f.shape[1]}×{H}×{W}",
                     tam_ancla=sizes, relaciones=ratios, k=k, salida_cls=f"{k}×{H}×{W}",
                     salida_reg=f"{d.shape[1]}×{H}×{W}", n_anclas=H * W * k))
display(pd.DataFrame(rows))
print(f"Total de anclas para esta imagen: {anchors[0].shape[0]:,}")

### 3.2 Anclas y parametrización de cajas

Para cada ancla $(x_a, y_a, w_a, h_a)$ y caja real $(x^*, y^*, w^*, h^*)$, la RPN aprende a predecir (Ec. 2 del paper):

$$t_x = \frac{x - x_a}{w_a},\quad t_y = \frac{y - y_a}{h_a},\quad t_w = \log\frac{w}{w_a},\quad t_h = \log\frac{h}{h_a}$$

Una ancla se etiqueta **positiva** si su IoU con alguna caja real es $\ge 0.7$ (o es la de mayor IoU para esa caja), **negativa** si su IoU es $< 0.3$ con todas, y se **ignora** en otro caso. En cada iteración se muestrean 256 anclas por imagen con proporción positivo:negativo de hasta 1:1.

In [ ]:
# Se toma el peatón más grande de la imagen y se dibujan las anclas centradas en él
img_resized = TF.resize(sample_img, list(images.image_sizes[0]), antialias=True)
gt_r = resize_boxes(sample_tgt["boxes"].to(DEVICE), sample_img.shape[-2:], images.image_sizes[0])
j = int(((gt_r[:, 2] - gt_r[:, 0]) * (gt_r[:, 3] - gt_r[:, 1])).argmax())
gt_box = gt_r[j]
cx, cy = float((gt_box[0] + gt_box[2]) / 2), float((gt_box[1] + gt_box[3]) / 2)

all_anchors = anchors[0]
ious = box_iou(all_anchors, gt_r)                        # [n_anclas, n_gt]
best_anchor = all_anchors[ious[:, j].argmax()]
max_iou_per_anchor = ious.max(dim=1).values
rpn = coco_model.rpn
# Regla del paper: positiva si IoU >= 0.7 o si es el ancla de mayor IoU para alguna caja real
pos_mask = max_iou_per_anchor >= rpn.proposal_matcher.high_threshold
pos_mask[ious.argmax(dim=0)] = True
neg_mask = (max_iou_per_anchor < rpn.proposal_matcher.low_threshold) & ~pos_mask
pos, neg = int(pos_mask.sum()), int(neg_mask.sum())
print(f"Umbrales RPN: positiva IoU≥{rpn.proposal_matcher.high_threshold} (o mejor ancla de cada caja), "
      f"negativa IoU<{rpn.proposal_matcher.low_threshold}")
print(f"Anclas positivas: {pos:,} | negativas: {neg:,} | ignoradas: {len(all_anchors) - pos - neg:,}")
print(f"Mini-lote de la RPN: {rpn.fg_bg_sampler.batch_size_per_image} anclas/imagen, "
      f"fracción positiva máx. {rpn.fg_bg_sampler.positive_fraction}")

# Codificación (t_x, t_y, t_w, t_h) de la caja real respecto a la mejor ancla, y decodificación inversa
t = rpn.box_coder.encode_single(gt_box[None], best_anchor[None])
decoded = rpn.box_coder.decode_single(t, best_anchor[None])
print("\nMejor ancla:", [round(v, 1) for v in best_anchor.tolist()], f"(IoU = {ious[:, j].max():.3f})")
print("Caja real:  ", [round(v, 1) for v in gt_box.tolist()])
print("Objetivos de regresión (t_x, t_y, t_w, t_h):", [round(v, 3) for v in t[0].tolist()])
print("Caja decodificada desde ancla + t:", [round(v, 1) for v in decoded[0].tolist()])

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
show_img(axes[0], img_resized, "Anclas (5 niveles × 3 relaciones) centradas en el peatón")
colors = plt.cm.viridis(np.linspace(0, 1, len(ag.cell_anchors)))
for lvl, (cell, c) in enumerate(zip(ag.cell_anchors, colors)):
    boxes_lvl = cell.cpu() + torch.tensor([cx, cy, cx, cy])
    draw_boxes(axes[0], boxes_lvl, c, lw=1.5)
axes[0].legend(handles=[mpatches.Patch(color=c, label=f"tamaño {s[0]}") for c, s in zip(colors, ag.sizes)],
               loc="lower right", fontsize=8)
show_img(axes[1], img_resized, "Mejor ancla (naranja) → regresión → caja real (verde)")
draw_boxes(axes[1], best_anchor[None].cpu(), "orange", lw=2.5)
draw_boxes(axes[1], gt_box[None].cpu(), "lime", ls="--", lw=2.5)
plt.tight_layout(); savefig("anclas"); plt.show()

### 3.3 Salida de la RPN: *objectness* y propuestas

El mapa de *objectness* muestra, para cada posición, la máxima probabilidad de objeto entre sus $k$ anclas. Luego las anclas se decodifican, se recortan al borde, se filtran con NMS (IoU 0.7) y se conservan las mejores $N$ como **propuestas** para el detector.

In [ ]:
with torch.no_grad():
    proposals, _ = coco_model.rpn(images, feats)   # en modo eval: propuestas ordenadas por objectness
props = proposals[0]

lvl = 1   # nivel P3 (stride 8)
H_img, W_img = images.image_sizes[0]
stride = images.tensors.shape[-1] / feat_list[lvl].shape[-1]
heat = objectness[lvl][0].sigmoid().max(dim=0).values
heat = heat[: math.ceil(H_img / stride), : math.ceil(W_img / stride)].cpu()

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
show_img(axes[0], img_resized, f"Objectness (nivel P{lvl + 2}, stride {stride:.0f})")
axes[0].imshow(heat.numpy(), cmap="jet", alpha=0.5, extent=(0, W_img, H_img, 0))
for ax, n in zip(axes[1:], [10, 100]):
    show_img(ax, img_resized, f"Top-{n} propuestas de la RPN (de {len(props)})")
    draw_boxes(ax, props[:n].cpu(), "cyan", lw=1)
    draw_boxes(ax, gt_r.cpu(), "lime", ls="--", lw=2.5)
plt.tight_layout(); savefig("rpn_objectness_propuestas"); plt.show()

### 3.4 Función de pérdida multitarea

$$L(\{p_i\},\{t_i\}) = \frac{1}{N_{cls}}\sum_i L_{cls}(p_i, p_i^*) + \lambda \frac{1}{N_{reg}} \sum_i p_i^* \, L_{reg}(t_i, t_i^*)$$

$L_{cls}$ es una pérdida logística (entropía cruzada) y $L_{reg}$ es **Smooth L1**, que solo se activa para anclas positivas ($p_i^*=1$). Torchvision reporta las cuatro pérdidas por separado:

| Clave | Módulo | Pérdida |
|---|---|---|
| `loss_objectness` | RPN | entropía cruzada binaria objeto/fondo |
| `loss_rpn_box_reg` | RPN | Smooth L1 sobre $(t_x,t_y,t_w,t_h)$ de las anclas |
| `loss_classifier` | Fast R-CNN | entropía cruzada sobre las clases (fondo, peatón) |
| `loss_box_reg` | Fast R-CNN | Smooth L1 del refinamiento de la caja |

La relación con el curso es directa: datos $X$ (mini-lote centrado en imagen), *forward* $\hat y$ (salidas `cls` y `reg`), pérdida $L$, *backpropagation* y actualización de parámetros con **SGD con momentum** y *weight decay*.

## 4. Métricas de evaluación

- **IoU** (*Intersection over Union*): $\mathrm{IoU}(A,B)=\frac{|A\cap B|}{|A\cup B|}$. Una detección es **verdadero positivo (TP)** si su IoU con una caja real todavía no emparejada es $\ge 0.5$.
- **Precision** $= \frac{TP}{TP+FP}$: de lo que el modelo detecta, cuánto es correcto.
- **Recall** $= \frac{TP}{TP+FN}$: de los peatones reales, cuántos se detectan.
- **F1** $= 2\frac{P\cdot R}{P+R}$.
- **AP** (*Average Precision*): área bajo la curva precision-recall al recorrer todos los umbrales de confianza (interpolación de 101 puntos, como COCO). **mAP** es el promedio de AP sobre las clases; con una sola clase, mAP = AP. Se reportan **AP@0.5** (criterio PASCAL VOC, el usado en el paper), **AP@0.75** (localización estricta) y **mAP@[0.5:0.95]** (promedio sobre 10 umbrales de IoU, criterio COCO).

**Taxonomía de errores** (inspirada en Hoiem et al., 2012). Cada detección con confianza ≥ umbral se clasifica como:
- **TP**: IoU ≥ 0.5 con un peatón no emparejado.
- **FP de localización**: 0.1 ≤ IoU máx. < 0.5 → encontró al peatón pero la caja es imprecisa.
- **FP duplicado**: IoU ≥ 0.5 con un peatón ya detectado (falla de NMS).
- **FP de fondo**: IoU < 0.1 con todo peatón → confundió el fondo con una persona.
- **FN**: peatón real sin ninguna detección emparejada.

In [ ]:
IOU_THRS = np.round(np.arange(0.5, 0.96, 0.05), 2)


def match_image(pred_boxes, pred_scores, gt_boxes, iou_thr=0.5):
    # Empareja detecciones con cajas reales en una imagen (voraz por confianza, estilo COCO)
    order = torch.argsort(pred_scores, descending=True)
    pb, ps = pred_boxes[order], pred_scores[order]
    P, G = len(pb), len(gt_boxes)
    is_tp = torch.zeros(P, dtype=torch.bool)
    matched_iou = torch.zeros(P)
    gt_matched = torch.zeros(G, dtype=torch.bool)
    kinds = []
    ious = box_iou(pb, gt_boxes) if P and G else torch.zeros(P, G)
    for i in range(P):
        max_iou = float(ious[i].max()) if G else 0.0
        cand = ious[i].clone() if G else torch.zeros(0)
        if G:
            cand[gt_matched] = -1.0
        jj = int(cand.argmax()) if G else -1
        if G and cand[jj] >= iou_thr:
            is_tp[i], gt_matched[jj], matched_iou[i] = True, True, cand[jj]
            kinds.append("TP")
        elif max_iou >= iou_thr:
            kinds.append("FP duplicado")
        elif max_iou >= 0.1:
            kinds.append("FP localización")
        else:
            kinds.append("FP fondo")
    return dict(boxes=pb, scores=ps, is_tp=is_tp, matched_iou=matched_iou, kinds=kinds, gt_matched=gt_matched)


def pr_curve(preds, gts, iou_thr=0.5):
    scores, tps, n_gt = [], [], 0
    for p, g in zip(preds, gts):
        m = match_image(p["boxes"], p["scores"], g, iou_thr)
        scores.append(m["scores"]); tps.append(m["is_tp"]); n_gt += len(g)
    scores = torch.cat(scores).numpy(); tps = torch.cat(tps).numpy()
    order = np.argsort(-scores, kind="stable")
    tps = tps[order]
    tp_c, fp_c = np.cumsum(tps), np.cumsum(~tps)
    recall = tp_c / max(n_gt, 1)
    precision = tp_c / np.maximum(tp_c + fp_c, 1e-9)
    return dict(precision=precision, recall=recall, scores=scores[order])


def average_precision(precision, recall):
    # AP con interpolación de 101 puntos de recall (COCO)
    if len(precision) == 0:
        return 0.0
    prec = np.maximum.accumulate(precision[::-1])[::-1]      # envolvente monótona
    idx = np.searchsorted(recall, np.linspace(0, 1, 101), side="left")
    return float(np.mean([prec[i] if i < len(prec) else 0.0 for i in idx]))


def analyze_errors(preds, gts, score_thr, iou_thr=0.5):
    counts, tp_ious, per_image = Counter(), [], []
    for p, g in zip(preds, gts):
        keep = p["scores"] >= score_thr
        m = match_image(p["boxes"][keep], p["scores"][keep], g, iou_thr)
        counts.update(m["kinds"])
        counts["FN"] += int((~m["gt_matched"]).sum())
        tp_ious += m["matched_iou"][m["is_tp"]].tolist()
        per_image.append(dict(boxes=m["boxes"], scores=m["scores"], kinds=m["kinds"],
                              fn_boxes=g[~m["gt_matched"]], gt=g))
    return counts, tp_ious, per_image


def prf_from_counts(c):
    tp = c["TP"]
    fp = c["FP localización"] + c["FP duplicado"] + c["FP fondo"]
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + c["FN"]) if tp + c["FN"] else 0.0
    f1 = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f1


def evaluate(preds, gts, score_thr=0.5):
    aps = {t: average_precision(**{k: v for k, v in pr_curve(preds, gts, t).items() if k != "scores"})
           for t in IOU_THRS}
    counts, tp_ious, _ = analyze_errors(preds, gts, score_thr, CFG["iou_thr"])
    p, r, f1 = prf_from_counts(counts)
    return {"AP50": aps[0.5], "AP75": aps[0.75], "mAP": float(np.mean(list(aps.values()))),
            "precision": p, "recall": r, "F1": f1,
            "IoU_medio_TP": float(np.mean(tp_ious)) if tp_ious else 0.0,
            **{k: counts[k] for k in ["TP", "FP localización", "FP duplicado", "FP fondo", "FN"]}}


# --- Pruebas unitarias de las métricas con casos sintéticos ---
g = torch.tensor([[0., 0., 10., 10.], [20., 20., 30., 30.]])
perfect = [dict(boxes=g.clone(), scores=torch.tensor([0.9, 0.8]))]
m = evaluate(perfect, [g])
assert abs(m["AP50"] - 1) < 1e-6 and abs(m["mAP"] - 1) < 1e-6 and m["precision"] == 1 and m["recall"] == 1
shifted = [dict(boxes=torch.tensor([[0., 0., 10., 10.], [0., 0., 10., 10.], [22., 22., 34., 34.], [50., 50., 60., 60.]]),
                scores=torch.tensor([0.9, 0.85, 0.8, 0.7]))]
c, _, _ = analyze_errors(shifted, [g], 0.5)
assert (c["TP"], c["FP duplicado"], c["FP localización"], c["FP fondo"], c["FN"]) == (1, 1, 1, 1, 1), c
assert abs(evaluate([dict(boxes=torch.zeros(0, 4), scores=torch.zeros(0))], [g])["AP50"]) < 1e-9
print("Pruebas de métricas: OK")

## 5. Entrenamiento y experimentos

### 5.1 Construcción de modelos (*transfer learning*)
Se parte de pesos preentrenados en **COCO** (que incluye la clase `person`) y se reemplaza el predictor final de la cabeza Fast R-CNN por uno de 2 clases (fondo, peatón). Se aplican las estrategias de la sesión 09:
- **Extractor congelado** (E1): `requires_grad=False` en todo el backbone + FPN; solo se entrenan RPN y cabeza.
- **Fine-tuning** (E2, E3): se descongelan las últimas 3 etapas del backbone (`trainable_backbone_layers=3`); las primeras capas (bordes, texturas genéricas) permanecen fijas.

Torchvision entrena con **entrenamiento conjunto aproximado** (Sec. 3.2-ii del paper): las cuatro pérdidas se suman y se optimizan a la vez. El experimento E4 implementa además el **entrenamiento alterno de 4 pasos** original.

In [ ]:
def build_model(arch="resnet50_fpn", num_classes=2, freeze_backbone=False, trainable_layers=3):
    kw = dict(min_size=CFG["min_size"], max_size=CFG["max_size"], trainable_backbone_layers=trainable_layers)
    if arch == "resnet50_fpn":
        model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1, **kw)
    elif arch == "mobilenet_v3_fpn":
        model = fasterrcnn_mobilenet_v3_large_fpn(weights=FasterRCNN_MobileNet_V3_Large_FPN_Weights.COCO_V1, **kw)
    else:
        raise ValueError(arch)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)   # nueva capa de salida
    if freeze_backbone:
        for p in model.backbone.parameters():
            p.requires_grad_(False)
    return model.to(DEVICE)


def to_device(images, targets):
    images = [im.to(DEVICE) for im in images]
    targets = [{k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in t.items()} for t in targets]
    return images, targets


@torch.no_grad()
def predict(model, dataset, keep_label=None, batch_size=4):
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn,
                        num_workers=CFG["num_workers"])
    preds, gts = [], []
    for images, targets in loader:
        outs = model([im.to(DEVICE) for im in images])
        for o, t in zip(outs, targets):
            b, s, l = o["boxes"].cpu(), o["scores"].cpu(), o["labels"].cpu()
            if keep_label is not None:
                k = l == keep_label
                b, s, l = b[k], s[k], l[k]
            preds.append(dict(boxes=b, scores=s, labels=l))
            gts.append(t["boxes"].cpu())
    return preds, gts


@torch.no_grad()
def get_proposals(model, img, top_n=None):
    # Propuestas de la RPN en coordenadas de la imagen original
    model.eval()
    images, _ = model.transform([img.to(DEVICE)])
    props, _ = model.rpn(images, model.backbone(images.tensors))
    p = props[0] if top_n is None else props[0][:top_n]
    return resize_boxes(p, images.image_sizes[0], img.shape[-2:]).cpu()


@torch.no_grad()
def proposal_recall(model, dataset, top_n=300):
    # Recall de las propuestas para cada umbral de IoU y Average Recall (AR) en [0.5:0.95]
    best = []
    for i in range(len(dataset)):
        img, tgt = dataset[i]
        props = get_proposals(model, img, top_n)
        if len(props) == 0:
            best.append(torch.zeros(len(tgt["boxes"])))
        else:
            best.append(box_iou(tgt["boxes"], props).max(dim=1).values)
    best = torch.cat(best)
    rec = {float(t): float((best >= t).float().mean()) for t in IOU_THRS}
    return rec, float(np.mean(list(rec.values())))

In [ ]:
def train_one_epoch(model, optimizer, loader, loss_keys=None, warmup=None):
    # Una época de SGD. loss_keys: subconjunto de pérdidas a optimizar (para el entrenamiento en 4 pasos)
    model.train()
    sums, n = Counter(), 0
    for images, targets in loader:
        images, targets = to_device(images, targets)
        loss_dict = model(images, targets)
        keys = loss_keys or list(loss_dict.keys())
        loss = sum(loss_dict[k] for k in keys)
        if not math.isfinite(loss.item()):
            raise RuntimeError(f"Pérdida no finita: {loss_dict}")
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if warmup is not None:
            warmup.step()
        for k, v in loss_dict.items():
            sums[k] += v.item()
        sums["loss_total"] += loss.item()
        n += 1
    return {k: v / n for k, v in sums.items()}


def fit(model, epochs, val_fn, loss_keys=None, tag=""):
    # Entrena con SGD + warmup lineal + StepLR, evalúa en validación cada época y conserva el mejor estado
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=CFG["lr"], momentum=CFG["momentum"], weight_decay=CFG["weight_decay"])
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=max(1, int(CFG["lr_step_frac"] * epochs)),
                                                gamma=CFG["lr_gamma"])
    history, best_score, best_state, best_epoch = [], -1.0, None, 0
    for epoch in range(1, epochs + 1):
        warmup = None
        if epoch == 1 and len(train_loader) > 1:
            warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.001,
                                                       total_iters=min(100, len(train_loader) - 1))
        lr = optimizer.param_groups[0]["lr"]
        t0 = time.time()
        losses = train_one_epoch(model, optimizer, train_loader, loss_keys, warmup)
        scheduler.step()
        val = val_fn(model)
        history.append(dict(epoch=epoch, lr=lr, **losses, **{f"val_{k}": v for k, v in val.items()},
                            seg=time.time() - t0))
        score = val["score"]
        print(f"[{tag}] época {epoch:2d}/{epochs} | pérdida {losses['loss_total']:.4f} | "
              + " | ".join(f"val {k} {v:.4f}" for k, v in val.items() if k != "score")
              + f" | {time.time() - t0:.0f}s")
        if score > best_score:
            best_score, best_epoch = score, epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    print(f"[{tag}] mejor época en validación: {best_epoch}")
    return pd.DataFrame(history), best_epoch


def val_detection(model):
    m = evaluate(*predict(model, val_ds))
    return {"AP50": m["AP50"], "mAP": m["mAP"], "score": m["mAP"]}


def val_rpn(model):
    _, ar = proposal_recall(model, val_ds, top_n=300)
    return {"AR@300": ar, "score": ar}

In [ ]:
RESULTS = {}
SCORE_THRS = np.round(np.arange(0.05, 0.96, 0.05), 2)


def finalize(name, model, desc, history=None, best_epoch=None, keep_label=None):
    # Evalúa en validación (para elegir el umbral de confianza que maximiza F1) y luego en prueba
    val_preds, val_gts = predict(model, val_ds, keep_label)
    f1s = [prf_from_counts(analyze_errors(val_preds, val_gts, t)[0])[2] for t in SCORE_THRS]
    thr = float(SCORE_THRS[int(np.argmax(f1s))])
    test_preds, test_gts = predict(model, test_ds, keep_label)
    RESULTS[name] = dict(
        model=model, desc=desc, history=history, best_epoch=best_epoch, thr=thr,
        val_metrics=evaluate(val_preds, val_gts, thr), test_metrics=evaluate(test_preds, test_gts, thr),
        test_metrics_05=evaluate(test_preds, test_gts, 0.5),
        val_preds=val_preds, val_gts=val_gts, test_preds=test_preds, test_gts=test_gts,
        params_total=count_params(model), params_trainable=count_params(model, trainable_only=True),
    )
    tm = RESULTS[name]["test_metrics"]
    print(f"{name}: umbral elegido en validación = {thr:.2f} | PRUEBA → AP50 {tm['AP50']:.3f} | "
          f"mAP {tm['mAP']:.3f} | P {tm['precision']:.3f} | R {tm['recall']:.3f} | F1 {tm['F1']:.3f}")

### 5.2 E0 — Línea base *zero-shot* (COCO, sin entrenamiento)
El modelo COCO detecta 91 categorías. Para evaluarlo como detector de peatones se conservan solo las detecciones de la clase `person` (etiqueta 1). Antes se revisa **qué otras clases predice** en las imágenes de prueba (análisis de predicciones de clases).

In [ ]:
raw_preds, _ = predict(coco_model, test_ds)
cls_counts = Counter(COCO_CLASSES[int(l)] for p in raw_preds for l, s in zip(p["labels"], p["scores"]) if s >= 0.5)
cls_df = pd.DataFrame(cls_counts.most_common(12), columns=["clase COCO", "# detecciones (score ≥ 0.5)"])
display(cls_df)
ax = cls_df.set_index("clase COCO").plot.barh(figsize=(8, 4), legend=False, color="#8172B2")
ax.invert_yaxis(); ax.set_title("E0: clases predichas por el modelo COCO en el conjunto de prueba")
plt.tight_layout(); savefig("e0_clases_coco"); plt.show()

finalize("E0_zero_shot_COCO", coco_model, "R50-FPN COCO sin entrenar (solo clase person)", keep_label=1)

### 5.3 E1, E2, E3 — *Fine-tuning* con entrenamiento conjunto

In [ ]:
EXPERIMENTS = [
    dict(name="E1_R50_extractor_congelado", arch="resnet50_fpn", freeze_backbone=True, trainable_layers=0,
         desc="R50-FPN: backbone+FPN congelados; se entrenan RPN y cabeza Fast R-CNN"),
    dict(name="E2_R50_finetuning", arch="resnet50_fpn", freeze_backbone=False, trainable_layers=3,
         desc="R50-FPN: fine-tuning de las 3 últimas etapas + RPN + cabeza (conjunto aproximado)"),
    dict(name="E3_MobileNetV3_finetuning", arch="mobilenet_v3_fpn", freeze_backbone=False, trainable_layers=3,
         desc="MobileNetV3-Large-FPN: fine-tuning (extractor ligero)"),
]

for exp in EXPERIMENTS:
    seed_everything()
    model = build_model(exp["arch"], freeze_backbone=exp["freeze_backbone"], trainable_layers=exp["trainable_layers"])
    print(f"\n=== {exp['name']} === parámetros entrenables: {count_params(model, True):,} / {count_params(model):,}")
    hist, best_ep = fit(model, CFG["epochs"], val_detection, tag=exp["name"])
    finalize(exp["name"], model, exp["desc"], hist, best_ep)
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

### 5.4 E4 — Entrenamiento alterno en 4 pasos (Sec. 3.2 del paper)

1. **Paso 1 — Entrenar la RPN** (red A): backbone + RPN se ajustan *end-to-end* solo con las pérdidas de la RPN.
2. **Paso 2 — Entrenar Fast R-CNN** (red B, independiente): su backbone y su cabeza se entrenan usando como entrada las propuestas **fijas** que genera la RPN del paso 1. Aún no hay capas compartidas.
3. **Paso 3 — Compartir y ajustar la RPN**: la RPN se monta sobre el backbone de B, que se **congela**; solo se ajustan las capas exclusivas de la RPN.
4. **Paso 4 — Ajustar el detector final**: con el backbone y la RPN fijos, solo se ajustan las capas exclusivas de Fast R-CNN.

El resultado es **una sola red** cuyas capas convolucionales son compartidas por la RPN y el detector.

> Adaptación: como en el resto de experimentos, ambas redes parten de pesos COCO (el paper parte de ImageNet) para que la comparación con E2 sea justa con solo 100 imágenes de entrenamiento.

In [ ]:
class FastRCNNWithExternalRPN(nn.Module):
    # Paso 2: detector Fast R-CNN (backbone + roi_heads de `det`) que usa propuestas FIJAS de otra red
    def __init__(self, det, rpn_net):
        super().__init__()
        self.det, self.rpn_net = det, rpn_net

    def forward(self, images, targets=None):
        original_sizes = [img.shape[-2:] for img in images]
        images_t, targets_t = self.det.transform(images, targets)
        with torch.no_grad():                         # la RPN del paso 1 no se entrena aquí
            self.rpn_net.eval()
            proposals, _ = self.rpn_net.rpn(images_t, self.rpn_net.backbone(images_t.tensors))
        feats = self.det.backbone(images_t.tensors)
        detections, losses = self.det.roi_heads(feats, proposals, images_t.image_sizes, targets_t)
        if self.training:
            return losses
        return self.det.transform.postprocess(detections, images_t.image_sizes, original_sizes)


def set_trainable(module, flag):
    for p in module.parameters():
        p.requires_grad_(flag)


RPN_LOSSES = ["loss_objectness", "loss_rpn_box_reg"]
DET_LOSSES = ["loss_classifier", "loss_box_reg"]
E = CFG["epochs_per_step_4pasos"]
hist_4 = {}

# Paso 1: red A = backbone + RPN entrenados solo con las pérdidas de la RPN
seed_everything()
net_a = build_model("resnet50_fpn", trainable_layers=3)
set_trainable(net_a.roi_heads, False)
hist_4["Paso 1: RPN"], _ = fit(net_a, E, val_rpn, loss_keys=RPN_LOSSES, tag="E4-paso1")
set_trainable(net_a, False)

# Paso 2: red B = detector Fast R-CNN independiente, entrenado con las propuestas fijas de A
net_b = build_model("resnet50_fpn", trainable_layers=3)
set_trainable(net_b.rpn, False)
step2 = FastRCNNWithExternalRPN(net_b, net_a)
hist_4["Paso 2: Fast R-CNN"], _ = fit(step2, E, val_detection, tag="E4-paso2")

# Paso 3: la RPN (inicializada desde A) se monta sobre el backbone de B, que queda congelado
net_b.rpn = copy.deepcopy(net_a.rpn)
set_trainable(net_b, False)
set_trainable(net_b.rpn, True)
hist_4["Paso 3: RPN compartida"], _ = fit(net_b, E, val_rpn, loss_keys=RPN_LOSSES, tag="E4-paso3")

# Paso 4: backbone y RPN fijos; solo se ajustan las capas exclusivas de Fast R-CNN
set_trainable(net_b, False)
set_trainable(net_b.roi_heads.box_head, True)
set_trainable(net_b.roi_heads.box_predictor, True)
hist_4["Paso 4: detector final"], _ = fit(net_b, E, val_detection, loss_keys=DET_LOSSES, tag="E4-paso4")

del net_a, step2
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
set_trainable(net_b, True)   # solo para contar parámetros totales; el modelo ya está entrenado
finalize("E4_R50_4_pasos", net_b, "R50-FPN: entrenamiento alterno de 4 pasos (paper, Sec. 3.2)",
         pd.concat([h.assign(paso=k) for k, h in hist_4.items()], ignore_index=True))
RESULTS["E4_R50_4_pasos"]["params_trainable"] = None

## 6. Resultados cuantitativos

### 6.1 Curvas de entrenamiento

In [ ]:
trained = [n for n in RESULTS if RESULTS[n]["history"] is not None and not n.startswith("E4")]
loss_keys = ["loss_objectness", "loss_rpn_box_reg", "loss_classifier", "loss_box_reg", "loss_total"]
titles = ["RPN: objectness", "RPN: regresión de cajas", "Fast R-CNN: clasificación",
          "Fast R-CNN: regresión de cajas", "Pérdida total"]
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for ax, k, ttl in zip(axes.ravel(), loss_keys, titles):
    for n in trained:
        h = RESULTS[n]["history"]
        ax.plot(h["epoch"], h[k], marker="o", label=n)
    ax.set(title=ttl, xlabel="época", ylabel="pérdida")
ax = axes.ravel()[-1]
for n in trained:
    h = RESULTS[n]["history"]
    ax.plot(h["epoch"], h["val_mAP"], marker="o", label=n)
ax.axhline(RESULTS["E0_zero_shot_COCO"]["val_metrics"]["mAP"], color="gray", ls="--", label="E0 zero-shot")
ax.set(title="mAP@[0.5:0.95] en validación", xlabel="época")
axes.ravel()[0].legend(fontsize=8)
ax.legend(fontsize=8)
plt.tight_layout(); savefig("curvas_entrenamiento"); plt.show()

# Entrenamiento en 4 pasos
h4 = RESULTS["E4_R50_4_pasos"]["history"]
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, (paso, h) in zip(axes, h4.groupby("paso", sort=False)):
    keys = RPN_LOSSES if "RPN" in paso else DET_LOSSES
    for k in keys:
        ax.plot(h["epoch"], h[k], marker="o", label=k)
    ax2 = ax.twinx()
    vk = "val_AR@300" if "RPN" in paso else "val_mAP"
    ax2.plot(h["epoch"], h[vk], color="k", ls="--", marker="s", label=vk)
    ax2.set_ylabel(vk)
    ax.set(title=paso, xlabel="época", ylabel="pérdida")
    ax.legend(fontsize=7, loc="upper left"); ax2.legend(fontsize=7, loc="upper right")
plt.suptitle("E4: entrenamiento alterno en 4 pasos", fontsize=13)
plt.tight_layout(); savefig("curvas_4_pasos"); plt.show()

### 6.2 Tabla comparativa (conjunto de prueba)
*Precision, Recall y F1* se calculan con el umbral de confianza que maximiza F1 en **validación** (columna `umbral`), para no ajustar nada sobre el conjunto de prueba. AP/mAP no dependen del umbral.

In [ ]:
rows = []
for n, r in RESULTS.items():
    m = r["test_metrics"]
    rows.append({"experimento": n, "AP@0.5": m["AP50"], "AP@0.75": m["AP75"], "mAP@[.5:.95]": m["mAP"],
                 "umbral": r["thr"], "Precision": m["precision"], "Recall": m["recall"], "F1": m["F1"],
                 "IoU medio (TP)": m["IoU_medio_TP"], "val mAP": r["val_metrics"]["mAP"],
                 "params (M)": r["params_total"] / 1e6,
                 "entrenables (M)": (r["params_trainable"] / 1e6 if r["params_trainable"] is not None else np.nan)
                 if r["history"] is not None else 0.0})
results_df = pd.DataFrame(rows).set_index("experimento")
results_df.to_csv(RESULTS_DIR / "metricas_prueba.csv")
display(results_df.style.format("{:.3f}").highlight_max(
    subset=["AP@0.5", "AP@0.75", "mAP@[.5:.95]", "F1", "IoU medio (TP)"], color="#c6efce"))

BEST = results_df["val mAP"].idxmax()      # el mejor modelo se elige por validación, no por prueba
print("Mejor modelo según validación:", BEST)

fig, ax = plt.subplots(figsize=(12, 4.5))
results_df[["AP@0.5", "AP@0.75", "mAP@[.5:.95]", "F1"]].plot.bar(ax=ax, rot=15)
ax.set(title="Comparación de experimentos (prueba)", ylim=(0, 1.05)); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); savefig("comparacion_metricas"); plt.show()

### 6.3 Curvas Precision–Recall y efecto del umbral de IoU

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for n, r in RESULTS.items():
    pr = pr_curve(r["test_preds"], r["test_gts"], 0.5)
    axes[0].plot(pr["recall"], pr["precision"], label=f"{n} (AP={r['test_metrics']['AP50']:.3f})")
    aps = [average_precision(**{k: v for k, v in pr_curve(r["test_preds"], r["test_gts"], t).items() if k != "scores"})
           for t in IOU_THRS]
    axes[1].plot(IOU_THRS, aps, marker="o", label=n)
axes[0].set(title="Curva Precision–Recall (IoU ≥ 0.5)", xlabel="Recall", ylabel="Precision", xlim=(0, 1.02), ylim=(0, 1.02))
axes[0].legend(fontsize=8)
axes[1].set(title="AP según el umbral de IoU exigido", xlabel="umbral de IoU", ylabel="AP")
axes[1].legend(fontsize=8)

# Barrido del umbral de confianza para el mejor modelo (en validación)
r = RESULTS[BEST]
prf = np.array([prf_from_counts(analyze_errors(r["val_preds"], r["val_gts"], t)[0]) for t in SCORE_THRS])
for i, lbl in enumerate(["Precision", "Recall", "F1"]):
    axes[2].plot(SCORE_THRS, prf[:, i], marker=".", label=lbl)
axes[2].axvline(r["thr"], color="k", ls="--", label=f"umbral elegido = {r['thr']:.2f}")
axes[2].set(title=f"{BEST}: barrido del umbral de confianza (validación)", xlabel="umbral de confianza")
axes[2].legend(fontsize=8)
plt.tight_layout(); savefig("curvas_pr"); plt.show()

**Lectura:** la caída de AP al exigir IoU más altos mide la **calidad de localización**: un modelo puede encontrar a casi todos los peatones (AP@0.5 alto) pero con cajas que no coinciden exactamente con el estilo de anotación del dataset (AP@0.75 menor). El *fine-tuning* suele mejorar sobre todo los umbrales altos, porque aprende la convención de anotación de Penn-Fudan.

## 7. Resultados cualitativos y análisis de errores

### 7.1 Predicciones de cada experimento sobre las mismas imágenes de prueba
Verde discontinuo: caja real · Rojo: predicción (con su confianza), usando el umbral elegido para cada modelo.

In [ ]:
n_show = min(4, len(test_ds))
show_ids = np.random.default_rng(SEED).choice(len(test_ds), n_show, replace=False)
names = list(RESULTS)
fig, axes = plt.subplots(n_show, len(names), figsize=(4.2 * len(names), 4.2 * n_show), squeeze=False)
for row, i in enumerate(show_ids):
    img, tgt = test_ds[int(i)]
    for col, n in enumerate(names):
        r = RESULTS[n]
        p = r["test_preds"][int(i)]
        keep = p["scores"] >= r["thr"]
        ax = axes[row, col]
        show_img(ax, img, n if row == 0 else "")
        draw_boxes(ax, tgt["boxes"], "lime", ls="--")
        draw_boxes(ax, p["boxes"][keep], "red", labels=[f"{s:.2f}" for s in p["scores"][keep].tolist()])
plt.tight_layout(); savefig("predicciones_comparadas"); plt.show()

### 7.2 Taxonomía de errores (prueba, umbral elegido en validación)

In [ ]:
err_cols = ["TP", "FP localización", "FP duplicado", "FP fondo", "FN"]
err_df = pd.DataFrame({n: {k: r["test_metrics"][k] for k in err_cols} for n, r in RESULTS.items()}).T
display(err_df)
err_colors = {"TP": "#2ca02c", "FP localización": "#ff7f0e", "FP duplicado": "#e377c2", "FP fondo": "#d62728", "FN": "#bcbd22"}
ax = err_df.plot.bar(figsize=(12, 4.5), rot=15, color=[err_colors[c] for c in err_cols])
ax.set(title="Tipos de acierto y error por experimento (prueba)", ylabel="# cajas"); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); savefig("taxonomia_errores"); plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
for n, r in RESULTS.items():
    _, tp_ious, _ = analyze_errors(r["test_preds"], r["test_gts"], r["thr"])
    ax.hist(tp_ious, bins=np.linspace(0.5, 1, 26), histtype="step", lw=2, label=f"{n} (media {np.mean(tp_ious) if tp_ious else 0:.3f})")
ax.set(title="Distribución del IoU de los verdaderos positivos", xlabel="IoU con la caja real", ylabel="# detecciones")
ax.legend(fontsize=8)
plt.tight_layout(); savefig("histograma_iou"); plt.show()

### 7.3 Galería de errores del mejor modelo
Azul: TP · Naranja: FP de localización · Rosa: FP duplicado · Rojo: FP de fondo · Amarillo grueso: FN (peatón no detectado) · Verde discontinuo: caja real.

In [ ]:
r = RESULTS[BEST]
_, _, per_image = analyze_errors(r["test_preds"], r["test_gts"], r["thr"])
kind_color = {"TP": "deepskyblue", "FP localización": "orange", "FP duplicado": "magenta", "FP fondo": "red"}
n_err = [sum(k != "TP" for k in pi["kinds"]) + len(pi["fn_boxes"]) for pi in per_image]
worst = [i for i in np.argsort(n_err)[::-1] if n_err[i] > 0][:6]
if worst:
    fig, axes = plt.subplots(1, len(worst), figsize=(4.5 * len(worst), 5.5), squeeze=False)
    for ax, i in zip(axes[0], worst):
        img, _ = test_ds[int(i)]
        pi = per_image[i]
        show_img(ax, img, f"prueba #{i}: {n_err[i]} error(es)")
        draw_boxes(ax, pi["gt"], "lime", ls="--", lw=1.5)
        for b, s, k in zip(pi["boxes"], pi["scores"], pi["kinds"]):
            draw_boxes(ax, b[None], kind_color[k], labels=[f"{k.replace('FP ', '')} {s:.2f}"])
        draw_boxes(ax, pi["fn_boxes"], "yellow", lw=4)
    plt.suptitle(f"Errores de {BEST} en prueba", fontsize=13)
    plt.tight_layout(); savefig("galeria_errores"); plt.show()
else:
    print("El mejor modelo no comete errores en prueba con el umbral elegido.")

**Para el análisis crítico:** revise los *FP de fondo* de la galería. En Penn-Fudan **no todas las personas están anotadas** (peatones lejanos, muy ocluidos o cortados por el borde de la imagen suelen quedar sin caja), así que parte de los “falsos positivos” son en realidad **aciertos del modelo penalizados por anotaciones incompletas**. Los *FP de localización* suelen deberse a oclusiones entre peatones y a diferencias en cómo se recortan pies y cabeza.

## 8. Complejidad computacional y número de propuestas

### 8.1 Tiempo de inferencia por componente
Se replica el análisis de tiempos del paper (Tabla 5): se mide por separado el preprocesamiento, el extractor de características, la RPN y la cabeza Fast R-CNN.

In [ ]:
@torch.no_grad()
def timed_breakdown(model, dataset, n_images=30, warmup=3):
    model.eval()
    ids = list(range(min(n_images, len(dataset))))
    for i in ids[:warmup]:                                   # calentamiento de la GPU
        model([dataset[i][0].to(DEVICE)])
    times = []
    for i in ids:
        img = dataset[i][0].to(DEVICE)
        sync(); t0 = time.perf_counter()
        images, _ = model.transform([img])
        sync(); t1 = time.perf_counter()
        feats = model.backbone(images.tensors)
        sync(); t2 = time.perf_counter()
        props, _ = model.rpn(images, feats)
        sync(); t3 = time.perf_counter()
        dets, _ = model.roi_heads(feats, props, images.image_sizes)
        model.transform.postprocess(dets, images.image_sizes, [img.shape[-2:]])
        sync(); t4 = time.perf_counter()
        times.append([t1 - t0, t2 - t1, t3 - t2, t4 - t3])
    ms = 1000 * np.array(times).mean(axis=0)
    return dict(zip(["Preprocesamiento", "Backbone + FPN (compartido)", "RPN (propuestas)", "Fast R-CNN (RoI + cabeza)"], ms))


timing_models = {"R50-FPN (E2)": "E2_R50_finetuning", "MobileNetV3-FPN (E3)": "E3_MobileNetV3_finetuning"}
timing_df = pd.DataFrame({lbl: timed_breakdown(RESULTS[n]["model"], test_ds) for lbl, n in timing_models.items()}).T
timing_df["Total (ms)"] = timing_df.sum(axis=1)
timing_df["FPS"] = 1000 / timing_df["Total (ms)"]
timing_df.to_csv(RESULTS_DIR / "tiempos_inferencia.csv")
display(timing_df.round(1))

ax = timing_df.drop(columns=["Total (ms)", "FPS"]).plot.barh(stacked=True, figsize=(12, 3.5),
                                                              color=["#bbbbbb", "#4C72B0", "#55A868", "#C44E52"])
ax.set(title=f"Tiempo de inferencia por imagen en {DEVICE.type.upper()} (ms)", xlabel="ms")
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout(); savefig("tiempos_inferencia"); plt.show()

### 8.2 Ablación: número de propuestas de la RPN en prueba
En el paper, la RPN logra buena precisión con solo **300 propuestas** (frente a 2000 de Selective Search), porque sus propuestas son de mayor calidad. Se varía el número $N$ de propuestas que la RPN entrega al detector y se mide el *recall* de las propuestas, el mAP y el tiempo.

In [ ]:
R50_BEST = max([n for n in RESULTS if "R50" in n], key=lambda n: RESULTS[n]["val_metrics"]["mAP"])
abl_model = RESULTS[R50_BEST]["model"]
orig_post = dict(abl_model.rpn._post_nms_top_n)
rows, recall_curves = [], {}
for N in [1000, 300, 100, 50, 20, 10]:
    abl_model.rpn._post_nms_top_n["testing"] = N
    rec, ar = proposal_recall(abl_model, test_ds, top_n=N)
    recall_curves[N] = rec
    sync(); t0 = time.perf_counter()
    preds, gts = predict(abl_model, test_ds)
    sync(); ms = 1000 * (time.perf_counter() - t0) / len(test_ds)
    m = evaluate(preds, gts)
    rows.append({"N propuestas": N, "Recall@0.5": rec[0.5], "Recall@0.7": rec[0.7], "AR [.5:.95]": ar,
                 "AP@0.5": m["AP50"], "mAP@[.5:.95]": m["mAP"], "ms/imagen": ms})
abl_model.rpn._post_nms_top_n.update(orig_post)
abl_df = pd.DataFrame(rows).set_index("N propuestas")
abl_df.to_csv(RESULTS_DIR / "ablacion_propuestas.csv")
print("Modelo:", R50_BEST)
display(abl_df.round(3))

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for N, rec in recall_curves.items():
    axes[0].plot(list(rec.keys()), list(rec.values()), marker="o", label=f"{N} propuestas")
axes[0].set(title=f"Recall de las propuestas vs. IoU ({R50_BEST})", xlabel="umbral de IoU", ylabel="Recall")
axes[0].legend(fontsize=8)
x = abl_df.index.astype(str)
axes[1].plot(x, abl_df["AP@0.5"], marker="o", label="AP@0.5")
axes[1].plot(x, abl_df["mAP@[.5:.95]"], marker="o", label="mAP@[.5:.95]")
axes[1].set(title="Precisión del detector según N", xlabel="N propuestas", ylim=(0, 1.02))
ax2 = axes[1].twinx()
ax2.bar(x, abl_df["ms/imagen"], alpha=0.2, color="gray", label="ms/imagen")
ax2.set_ylabel("ms / imagen")
axes[1].legend(loc="lower right", fontsize=8)
plt.tight_layout(); savefig("ablacion_propuestas"); plt.show()

## 9. Conclusiones

La siguiente celda genera un resumen automático con los números de **su** ejecución, para redactar las conclusiones de la presentación.

In [ ]:
e0 = RESULTS["E0_zero_shot_COCO"]["test_metrics"]
b = RESULTS[BEST]["test_metrics"]
summary = {
    "dataset": {"train": len(train_ds), "val": len(val_ds), "test": len(test_ds), "peatones_total": int(len(boxes_df))},
    "mejor_modelo_validacion": BEST,
    "metricas_prueba": results_df.round(4).to_dict(orient="index"),
    "tiempos_ms": timing_df.round(2).to_dict(orient="index"),
    "ablacion_propuestas": abl_df.round(4).to_dict(orient="index"),
    "config": CFG, "quick_run": QUICK_RUN, "device": str(DEVICE),
}
with open(RESULTS_DIR / "resumen.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2, default=str)

def delta(k):
    return f"{e0[k]:.3f} → {b[k]:.3f} ({b[k] - e0[k]:+.3f})"

e1, e2 = RESULTS["E1_R50_extractor_congelado"]["test_metrics"], RESULTS["E2_R50_finetuning"]["test_metrics"]
e4 = RESULTS["E4_R50_4_pasos"]["test_metrics"]
fast, slow = timing_df["FPS"].idxmax(), timing_df["FPS"].idxmin()
print(f"• Mejor modelo (elegido en validación): {BEST}")
print(f"• Zero-shot → mejor modelo: AP@0.5 {delta('AP50')} | AP@0.75 {delta('AP75')} | mAP {delta('mAP')}")
print(f"• Extractor congelado (E1) vs. fine-tuning (E2): mAP {e1['mAP']:.3f} vs {e2['mAP']:.3f}")
print(f"• Entrenamiento 4 pasos (E4) vs. conjunto (E2): mAP {e4['mAP']:.3f} vs {e2['mAP']:.3f}")
print(f"• Velocidad: {fast} a {timing_df.loc[fast, 'FPS']:.1f} FPS vs. {slow} a {timing_df.loc[slow, 'FPS']:.1f} FPS")
print(f"• Errores del mejor modelo: " + ", ".join(f"{k}={b[k]}" for k in err_cols))
print(f"• Con {abl_df.index[1]} propuestas: AP@0.5 {abl_df.iloc[1]['AP@0.5']:.3f} "
      f"(con {abl_df.index[0]}: {abl_df.iloc[0]['AP@0.5']:.3f})")
print(f"\nArchivos generados en {RESULTS_DIR.resolve()}:")
for p in sorted(RESULTS_DIR.iterdir()):
    print("  -", p.name)

### Guía para la discusión (completar con los números del resumen anterior)

1. **Transfer learning:** comparar E0 (zero-shot) con los modelos ajustados. ¿El ajuste mejora más AP@0.5 (encontrar peatones) o AP@0.75 (localizarlos con precisión)? Un modelo COCO ya sabe qué es una persona; lo que aprende con *fine-tuning* es principalmente la **convención de anotación** de Penn-Fudan y el dominio (escenas urbanas, peatones de pie).
2. **Capas compartidas congeladas vs. ajustadas (E1 vs. E2):** cuantificar cuánto aporta ajustar el extractor y el costo en parámetros entrenables.
3. **Entrenamiento de 4 pasos vs. conjunto (E4 vs. E2):** el paper reporta resultados similares (Sec. 3.2) y el entrenamiento conjunto reduce el tiempo de entrenamiento; contrastar con lo obtenido.
4. **Velocidad vs. precisión (E2 vs. E3):** relacionarlo con el *trade-off* ZF vs. VGG del paper. La RPN aporta una fracción mínima del tiempo total: es el resultado central del artículo (propuestas “casi gratis”).
5. **Número de propuestas:** ¿cuántas propuestas se necesitan para no perder precisión? Comparar con las 300 del paper.
6. **Errores:** qué tipo domina (localización, fondo, FN) y por qué (oclusiones, peatones pequeños o cortados, anotaciones incompletas).

**Limitaciones:** dataset pequeño (170 imágenes) y de una sola clase; partición propia (Penn-Fudan no tiene una oficial); la evaluación con 50 imágenes de prueba tiene varianza alta; el backbone R50-FPN de torchvision difiere del VGG-16 del paper; las anotaciones incompletas penalizan detecciones correctas.

## 10. Referencias
- Ren, S., He, K., Girshick, R., & Sun, J. (2015). *Faster R-CNN: Towards Real-Time Object Detection with Region Proposal Networks*. NeurIPS. arXiv:1506.01497.
- Girshick, R. (2015). *Fast R-CNN*. ICCV.
- Lin, T.-Y., Dollár, P., Girshick, R., He, K., Hariharan, B., & Belongie, S. (2017). *Feature Pyramid Networks for Object Detection*. CVPR.
- Howard, A. et al. (2019). *Searching for MobileNetV3*. ICCV.
- Wang, L., Shi, J., Song, G., & Shen, I. (2007). *Object Detection Combining Recognition and Segmentation*. ACCV (dataset Penn-Fudan).
- Hoiem, D., Chodpathumwan, Y., & Dai, Q. (2012). *Diagnosing Error in Object Detectors*. ECCV.
- Tutorial oficial de PyTorch: [TorchVision Object Detection Finetuning Tutorial](https://pytorch.org/tutorials/intermediate/torchvision_tutorial.html).
- Documentación de modelos de detección de torchvision: [torchvision.models.detection](https://pytorch.org/vision/stable/models.html#object-detection).